# Baseline Models

This notebook evaluates simple baseline models for the sticker sales forecasting task using the shared temporal validation and MAPE metric.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.baselines import evaluate_baselines, load_training_data, save_metrics

## Load Data

In [ ]:
train_path = PROJECT_ROOT / "data" / "train.csv"
metrics_path = PROJECT_ROOT / "results" / "metrics.csv"

train = load_training_data(train_path)
train.shape

## Evaluate Baselines

In [ ]:
metrics = evaluate_baselines(train, include_long_horizon=True)
save_metrics(metrics, metrics_path)
metrics

## Summary

In [ ]:
summary = (
    metrics.groupby(["model", "validation_scheme"], as_index=False)["mape"]
    .mean()
    .sort_values(["validation_scheme", "mape"])
)

summary

In [ ]:
plot_data = summary[summary["validation_scheme"] == "expanding_window"]

plt.figure(figsize=(8, 4))
plt.bar(plot_data["model"], plot_data["mape"])
plt.ylabel("MAPE, %")
plt.title("Baseline comparison on expanding-window validation")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## Notes

- `group_median` checks how much can be explained by average sales level for each country-store-product group.
- `seasonal_naive_last_year` uses the same date from the previous year and falls back to group median when the exact value is unavailable.
- `ridge_calendar_one_hot` is a simple ML baseline with calendar features and one-hot encoded categorical variables.